# Team Performance Analytics

This notebook analyzes team member performance metrics using data from Google Sheets. We'll create various visualizations to understand:
- Daily performance patterns
- Weekly trends and patterns
- Monthly performance overview
- Key Performance Indicators (KPIs)

First, let's set up our environment and import the required libraries.

In [ ]:
# Team Performance Analytics Dashboard
# =====================================
# This notebook analyzes team member performance metrics from Google Sheets

# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Google Sheets Configuration
SPREADSHEET_ID = "1KDj8ixj9oF5jSFEh-KXakLgJXjTwkgXn8uiHdxajaOc"
CREDENTIALS_PATH = "../src/config/credentials.json"

# Initialize Google Sheets API
print("Connecting to Google Sheets...")
creds = Credentials.from_service_account_file(
    CREDENTIALS_PATH,
    scopes=['https://www.googleapis.com/auth/spreadsheets.readonly']
)
service = build('sheets', 'v4', credentials=creds)

# Get spreadsheet metadata to find all sheets
spreadsheet = service.spreadsheets().get(spreadsheetId=SPREADSHEET_ID).execute()
sheets_list = spreadsheet.get('sheets', [])

print(f"Found {len(sheets_list)} team member sheets\n")

# Fetch data from all team member sheets
all_data = []

for sheet in sheets_list:
    sheet_name = sheet['properties']['title']
    print(f"Fetching data from: {sheet_name}")
    
    # Get data from each sheet
    range_name = f"{sheet_name}!A2:M"
    result = service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID,
        range=range_name
    ).execute()
    
    values = result.get('values', [])
    
    # Process each row
    for row in values:
        if len(row) >= 13:
            ticket_data = {
                'Member': sheet_name,
                'Ticket No': row[0],
                'Created Date': row[1],
                'Severity': row[2],
                'SLA Hours': int(row[3]) if row[3] else 0,
                'SLA Deadline': row[4],
                'Status': row[5],
                'Assigned Level': row[6],
                'Assigned To': row[7],
                'Resolution Date': row[8],
                'Carry Forward': row[9],
                'Days Forwarded': int(row[10]) if row[10] else 0,
                'Remarks': row[11],
                'Urgency Score': int(row[12]) if row[12] else 0
            }
            all_data.append(ticket_data)

# Create DataFrame
df_combined = pd.DataFrame(all_data)

print(f"\nTotal tickets loaded: {len(df_combined)}")
print(f"Team members: {df_combined['Member'].nunique()}")

print("Data loaded successfully!")
df_combined.head()

# Daily Performance Analysis

Let's create visualizations to analyze daily performance metrics for each team member.

In [ ]:
# Daily Tickets Analysis - Bar Chart
plt.figure(figsize=(15, 6))

# Group by date and member, calculate mean for assigned and resolved tickets
daily_tickets = df_combined.groupby(['Date', 'Member']).agg({
    'Tickets Assigned': 'sum',
    'Tickets Resolved': 'sum'
}).reset_index()

# Create bar chart using plotly
fig = px.bar(daily_tickets, x='Date', y=['Tickets Assigned', 'Tickets Resolved'],
             color='Member', barmode='group',
             title='Daily Tickets Assigned vs Resolved by Team Member',
             labels={'value': 'Number of Tickets', 'variable': 'Metric'})

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Number of Tickets",
    legend_title="Team Member & Metric",
    template='plotly_white'
)

fig.show()

# Daily SLA Breaches Trend
plt.figure(figsize=(15, 6))

daily_sla = df_combined.groupby(['Date', 'Member'])['SLA Breaches'].sum().reset_index()

fig = px.line(daily_sla, x='Date', y='SLA Breaches', color='Member',
              title='Daily SLA Breaches Trend by Team Member',
              labels={'SLA Breaches': 'Number of SLA Breaches'})

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Number of SLA Breaches",
    template='plotly_white'
)

fig.show()

# Stacked Bar: Responded Tickets vs Client Interactions
plt.figure(figsize=(15, 6))

daily_interactions = df_combined.groupby(['Date', 'Member']).agg({
    'Resopond Tickets': 'sum',
    'Client Interactions': 'sum'
}).reset_index()

fig = px.bar(daily_interactions, x='Date', 
             y=['Resopond Tickets', 'Client Interactions'],
             color='Member', 
             title='Daily Responded Tickets vs Client Interactions',
             barmode='stack')

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Count",
    legend_title="Team Member",
    template='plotly_white'
)

fig.show()

# Weekly Performance Analysis

Now let's analyze the weekly patterns in team performance.

# Team Performance Analytics Dashboard

This notebook analyzes team member performance metrics using data from Google Sheets. We'll create various visualizations to understand daily, weekly, and monthly patterns in:
- Ticket assignments and resolutions
- SLA breaches
- Client interactions
- Response times

## Setup and Dependencies

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from datetime import datetime, timedelta
import calendar

# Set style for matplotlib
plt.style.use('seaborn')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = [12, 6]

In [ ]:
# Load and preprocess data
def load_and_preprocess_data(spreadsheet_id):
    # Load data from Google Sheets
    # For now, we'll use sample data
    data = pd.DataFrame({
        'Date': pd.date_range(start='2025-01-01', periods=100),
        'Name': np.random.choice(['Winnish Allwin G J', 'Vuppu Reddy Kalyan', 'Sam J', 'Kiran Kumar'], 100),
        'Email': ['email@example.com'] * 100,
        'Tickets_Assigned': np.random.randint(5, 20, 100),
        'Tickets_Resolved': np.random.randint(3, 15, 100),
        'SLA_Breaches': np.random.randint(0, 5, 100),
        'Respond_Tickets': np.random.randint(2, 10, 100),
        'Client_Interaction': np.random.randint(1, 8, 100),
        'Remarks': np.random.choice(['Bug Fix', 'Feature Request', 'Support', 'Documentation'], 100)
    })
    
    # Convert date column to datetime
    data['Date'] = pd.to_datetime(data['Date'])
    
    # Add derived columns
    data['Week'] = data['Date'].dt.isocalendar().week
    data['Month'] = data['Date'].dt.month
    data['Day_of_Week'] = data['Date'].dt.day_name()
    data['Resolution_Rate'] = (data['Tickets_Resolved'] / data['Tickets_Assigned'] * 100).round(2)
    
    return data

# Load the data
df = load_and_preprocess_data("your_spreadsheet_id")